# Kaggle launcher: one run per session

This notebook holds no project logic. It clones the repository, installs it and calls the scripts.

**Setup:** Accelerator = **GPU T4 x2**, Internet = **on** (Session options, right-hand panel).

**How to use:**
1. Set the values in the first code cell.
2. *Save Version* -> *Save & Run All* (runs in the background, up to Kaggle's session limit).
3. When it finishes, download `<run_id>.zip` from the Output tab and commit its files to `results/runs/<run_id>/`. Update the run's row in `results/run_log.csv` from `run_info.json`.

| What | MODE | ARM / RUN_ID |
| --- | --- | --- |
| Smoke test | `train+eval` | each arm in turn, `CONFIG = "configs/smoke.yaml"` |
| Pilot (T3) | `train+eval` | `RUN_ID = "pilot_<arm>_s3"` |
| Main run | `train+eval` | `ARM`, `SEED`; `RUN_ID = None` gives `<arm>_s<seed>` |
| References (E2) | `eval` | `RUN_ID = "ref_teacher"` or `"ref_student0"` |

In [ ]:
# ---- Settings: the only cell you edit ----
REPO = "https://github.com/redkitex/Knowledge-Distillation-and-Model-Compression.git"
REF = "main"                    # branch or commit; real runs use the commit of the config lock
CONFIG = "configs/smoke.yaml"   # configs/default.yaml for real runs
MODE = "train+eval"             # "train+eval" or "eval" (references only)
ARM = "human"                   # human | teacher | onpolicy | ref_hardlabel
SEED = 1
RUN_ID = None                   # None -> f"{ARM}_s{SEED}"; e.g. "pilot_human_s3", "ref_teacher"
ACCOUNT = "T3"                  # whose Kaggle account this is, for the run log

In [ ]:
# Private repository? Add a Kaggle secret named GITHUB_TOKEN (Add-ons -> Secrets) and uncomment:
# from kaggle_secrets import UserSecretsClient
# REPO = REPO.replace("https://", "https://" + UserSecretsClient().get_secret("GITHUB_TOKEN") + "@")

# Optional, for faster Hugging Face downloads: a secret named HF_TOKEN.
# import os
# os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

In [ ]:
import torch

run_id = RUN_ID or f"{ARM}_s{SEED}"
n_gpus = torch.cuda.device_count()
print("run:", run_id, "| mode:", MODE, "| GPUs:", n_gpus)
if MODE == "train+eval":
    assert n_gpus >= 2, "Training needs Accelerator = GPU T4 x2 (teacher on GPU 1, student on GPU 0)"
else:
    assert n_gpus >= 1, "Evaluation needs a GPU"

In [ ]:
%cd /kaggle/working
!rm -rf repo && git clone -q {REPO} repo
%cd /kaggle/working/repo
!git checkout -q {REF} && git log -1 --oneline
!pip install -q -r requirements.txt && pip install -q -e .

In [ ]:
if MODE == "train+eval":
    !python scripts/20_train.py --config {CONFIG} --arm {ARM} --seed {SEED} --run-id {run_id} --account {ACCOUNT}
!python scripts/30_evaluate.py --config {CONFIG} --run {run_id}

In [ ]:
# Package the small outputs for download. The checkpoint stays in this session's
# output (results/runs/<run_id>/checkpoint) and never goes into git.
import shutil

src = f"results/runs/{run_id}"
dst = f"/kaggle/working/{run_id}"
shutil.copytree(src, dst, ignore=shutil.ignore_patterns("checkpoint"), dirs_exist_ok=True)
shutil.make_archive(dst, "zip", dst)
print("Download from the Output tab:", dst + ".zip")
!ls -la {dst}